# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [8]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvidia-smi: command not found


In [6]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

SecretNotFoundError: Secret KAGGLE_USERNAME does not exist.

### 0.1. Mount Google Drive
Kết nối với Google Drive để sao lưu và bảo toàn các checkpoints cũng như LoRA adapters sau khi huấn luyện.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 0.2. Kiểm tra tính toàn vẹn của mã nguồn bài tập (Unit Tests)
Chạy bộ kiểm thử tự động bằng `pytest` để đảm bảo môi trường và cấu trúc các hàm bổ trợ sẵn có hoạt động chính xác.

In [ ]:
!git clone -q https://github.com/nhat-thang/K4-Track-4-Day10-VisionLanguageModel /content/lab10
%cd /content/lab10
!ls

In [ ]:
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Không dùng tài khoản Kaggle: tải zip công khai qua URL
!python scripts/prepare_data.py --zip "https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes"

In [ ]:
# Bước 1: đọc split, thống kê, kiểm tra session không trùng giữa các split, vẽ 6 ảnh có box
import json, random
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from scripts.labcommon import read_jsonl, CLASSES

DATA = Path("data")
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}

for s, rows in splits.items():
    cnt = Counter(o["label"] for r in rows for o in r["objects"])
    n_empty = sum(1 for r in rows if not r["objects"])
    print(f"{s:5s} {len(rows):5d} ảnh | {sum(cnt.values()):5d} box {dict(cnt)} | ảnh không có box: {n_empty}")

# Không session nào được xuất hiện ở hai split
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = sessions[a] & sessions[b]
    print(f"session chung {a}/{b}: {len(inter)}")
    assert not inter, inter
print({s: len(v) for s, v in sessions.items()}, "session mỗi split")
print("Mẫu:", json.dumps(splits["train"][0], ensure_ascii=False)[:300])

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def show(rows, n=6, seed=0):
    rows = [r for r in rows if r["objects"]]
    picks = random.Random(seed).sample(rows, n)
    fig, axes = plt.subplots(2, 3, figsize=(18, 7))
    for ax, r in zip(axes.flat, picks):
        img = Image.open(DATA / r["image_path"])
        W, H = img.size
        ax.imshow(img); ax.axis("off"); ax.set_title(r["image_id"][:40], fontsize=8)
        for o in r["objects"]:
            xc, yc, w, h = o["yolo"]
            x0, y0 = (xc - w / 2) * W, (yc - h / 2) * H
            c = COLORS[o["label"]]
            ax.add_patch(patches.Rectangle((x0, y0), w * W, h * H, fill=False, ec=c, lw=2))
            ax.text(x0, y0, o["label"], fontsize=7, bbox=dict(fc=c, alpha=0.7, pad=1))
    plt.tight_layout(); plt.show()

show(splits["train"])

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Bước 2: nhãn YOLO -> JSON thang 0–1000 + prompt
import json
from scripts.labcommon import CLASSES

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo                                   # chuẩn hoá [0,1], tâm + kích thước
    clamp = lambda v: max(0, min(1000, int(round(v * 1000))))
    x1, y1 = clamp(xc - w / 2), clamp(yc - h / 2)
    x2, y2 = clamp(xc + w / 2), clamp(yc + h / 2)
    # box quá nhỏ có thể bị làm tròn thành x2 <= x1 -> score.py coi là JSON hỏng; nới ra 1 đơn vị
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < 1000 else (999, 1000)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < 1000 else (999, 1000)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))   # trên→dưới, trái→phải
    return json.dumps(boxes, separators=(",", ":"))                # "[]" nếu không có object

PROMPT = (
    "Detect all road damage in this image. Classes: pothole, crack, manhole. "
    "Return ONLY a JSON list, no other text, in the form "
    '[{"label":"pothole","bbox_2d":[x1,y1,x2,y2]}]. '
    "Coordinates are integers on a 0-1000 scale relative to image width and height "
    "(x1,y1 = top-left, x2,y2 = bottom-right). Return [] if there is no damage."
)

# Kiểm tra: JSON -> pixel, so với box gốc; vẽ chồng lên nhau
r = next(r for r in splits["train"] if len(r["objects"]) >= 2)
ans = objects_to_answer(r["objects"])
print(ans)
img = Image.open(DATA / r["image_path"]); W, H = img.size
fig, ax = plt.subplots(figsize=(10, 6)); ax.imshow(img); ax.axis("off")
for o in r["objects"]:
    xc, yc, w, h = o["yolo"]
    g = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
    ax.add_patch(patches.Rectangle(g[:2], g[2] - g[0], g[3] - g[1], fill=False, ec="lime", lw=4))
for b in json.loads(ans):
    x1, y1, x2, y2 = b["bbox_2d"]
    p = (x1 / 1000 * W, y1 / 1000 * H, x2 / 1000 * W, y2 / 1000 * H)
    ax.add_patch(patches.Rectangle(p[:2], p[2] - p[0], p[3] - p[1], fill=False, ec="red", lw=1.5, ls="--"))
    ax.text(p[0], p[1], b["label"], color="red")
plt.title("xanh = box gốc (YOLO), đỏ đứt = JSON 0–1000 quy ngược về pixel"); plt.show()

# Toàn bộ train: mọi đáp án phải parse được bằng score.py và có nhãn hợp lệ
from scripts.score import parse_output
bad = [r["image_id"] for r in splits["train"] if parse_output(objects_to_answer(r["objects"])) is None]
print("đáp án train không parse được:", len(bad))
assert not bad and all(o["label"] in CLASSES for r in splits["train"] for o in r["objects"])
print(PROMPT)

## 3. E0: zero-shot

In [ ]:
# Bước 3: E0 zero-shot — nạp Qwen3-VL-4B 4-bit, suy luận trên test -> work/e0.jsonl
import os, json, time, shutil, torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from scripts.labcommon import read_jsonl
transformers_logging = __import__("transformers").logging
transformers_logging.set_verbosity_error()                 # tắt cảnh báo processor_kwargs lặp lại mỗi batch

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
BACKUP = Path("/content/drive/MyDrive/lab10_backup")      # chép kết quả lên Drive phòng mất session

def load_base_model():
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["model.visual", "lm_head"])   # giữ ViT + projector + lm_head ở fp16
        # (transformers 5.x so khớp từ ĐẦU tên module: "visual" không khớp "model.visual.*")
    m = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={"": 0})
    m.eval()
    return m

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"                   # left pad khi generate theo batch
model = load_base_model()
print(f"VRAM sau khi nạp: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
vit_types = {type(mod).__name__ for n, mod in model.named_modules() if n.startswith("model.visual") and hasattr(mod, "weight")}
print("kiểu layer trong ViT:", vit_types)
assert "Linear4bit" not in vit_types, "ViT bị lượng tử hoá 4-bit!"

def build_messages(img, prompt=None):
    return [{"role": "user", "content": [{"type": "image", "image": img},
                                         {"type": "text", "text": prompt or PROMPT}]}]

@torch.inference_mode()
def predict(model, rows, out_path, bs=4, max_new_tokens=256, prompt=None):
    """Suy luận và ghi từng dòng {"image_id","raw"}; chạy lại sẽ bỏ qua ảnh đã có (resume)."""
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    done = {r["image_id"] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r["image_id"] not in done]
    print(f"{out_path}: đã có {len(done)}, cần chạy {len(todo)}")
    t0 = time.time()
    with open(out_path, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), bs)):
            batch = todo[i:i + bs]
            msgs = [build_messages(Image.open(DATA / r["image_path"]).convert("RGB"), prompt) for r in batch]
            inputs = processor.apply_chat_template(
                msgs, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for r, t in zip(batch, texts):
                f.write(json.dumps({"image_id": r["image_id"], "raw": t}, ensure_ascii=False) + "\n")
            f.flush()
    print(f"xong trong {(time.time() - t0) / 60:.1f} phút")
    if BACKUP.parent.exists():
        BACKUP.mkdir(exist_ok=True); shutil.copy(out_path, BACKUP / out_path.name)

# Thử nhanh 1 ảnh trước khi chạy cả test
predict(model, splits["test"][:1], "work/_smoke.jsonl")
print(read_jsonl("work/_smoke.jsonl")[-1]["raw"])

predict(model, splits["test"], "work/e0.jsonl")

# In 5 output thô để xem model trả gì (có ```json không? chữ thừa? thang toạ độ?)
for r in read_jsonl("work/e0.jsonl")[:5]:
    print(r["image_id"], "->", r["raw"][:300].replace("\n", " "), "\n")

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Bước 4: ví dụ huấn luyện (chat template) + mask loss + collate
import torch.nn.functional as F

def encode_example(img, prompt, answer):
    user = build_messages(img, prompt)[0]
    asst = {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    kw = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt_enc = processor.apply_chat_template([user], add_generation_prompt=True, **kw)
    full_enc = processor.apply_chat_template([user, asst], add_generation_prompt=False, **kw)
    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), "prompt không phải tiền tố"
    ex = {k: v for k, v in full_enc.items()}
    labels = ex["input_ids"].clone()
    labels[:, :n] = -100                                  # chỉ học phần trả lời
    ex["labels"] = labels
    return ex

NO_PAD = ("pixel_values", "image_grid_thw")               # nối theo chiều 0, KHÔNG pad
PAD_VALUE = {"input_ids": processor.tokenizer.pad_token_id, "labels": -100}  # còn lại (attention_mask, mm_token_type_ids...) pad 0

def collate(batch):
    L = max(b["input_ids"].shape[1] for b in batch)
    out = {}
    for k in batch[0]:
        if k in NO_PAD:
            out[k] = torch.cat([b[k] for b in batch], dim=0)
        else:                                             # pad phải đến cùng độ dài
            out[k] = torch.cat([F.pad(b[k], (0, L - b[k].shape[1]), value=PAD_VALUE.get(k, 0)) for b in batch], dim=0)
    return out

def make_example(row, prompt=None, answer_fn=None):
    img = Image.open(DATA / row["image_path"]).convert("RGB")
    return encode_example(img, prompt or PROMPT, (answer_fn or objects_to_answer)(row["objects"]))

# Kiểm tra
ex = make_example(splits["train"][0])
img_tok = model.config.image_token_id
print("keys:", list(ex.keys()))
print("kích thước ảnh:", splits["train"][0]["width"], "x", splits["train"][0]["height"],
      "| số token ảnh:", int((ex["input_ids"] == img_tok).sum()), "(kỳ vọng 220 cho 640x360)")
print("token có loss:", repr(processor.tokenizer.decode(ex["labels"][0][ex["labels"][0] != -100])))
print("đáp án gốc  :", repr(objects_to_answer(splits["train"][0]["objects"])))
b = collate([make_example(r) for r in splits["train"][:2]])
print({k: tuple(v.shape) for k, v in b.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Bước 5: setup LoRA / đóng băng cho E1, E2, E3
import re
from peft import LoraConfig, get_peft_model

LLM_RE = r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
VIT_RE = r"model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.linear_fc[12])"
PROJ_KEYS = ("visual.merger", "visual.deepstack_merger_list")

def apply_setup(model, setup, r=16, alpha=32, dropout=0.05):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()                    # cần cho gradient checkpointing
    pattern = LLM_RE if setup in ("E1", "E2") else f"(?:{LLM_RE}|{VIT_RE})"
    matched = [n for n, _ in model.named_modules() if re.fullmatch(pattern, n)]
    print(f"[{setup}] số module LoRA: {len(matched)}  (kỳ vọng {252 if setup in ('E1', 'E2') else 348})")
    pm = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                                          target_modules=pattern))
    extra = []                                            # tham số projector huấn luyện đầy đủ (E2, E3)
    if setup in ("E2", "E3"):
        for n, p in pm.named_parameters():
            if any(k in n for k in PROJ_KEYS):
                p.data = p.data.float()
                p.requires_grad = True
                extra.append(n)
    n_train = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    n_extra = sum(p.numel() for n, p in pm.named_parameters() if n in extra)
    print(f"[{setup}] tham số huấn luyện: {n_train / 1e6:.2f}M (projector: {n_extra / 1e6:.2f}M, {len(extra)} tensor)")
    return pm, extra, n_train

# Kiểm tra nhanh số module khớp trên model E0 đang có (chỉ đếm, không sửa model)
for s, pat in [("E1/E2", LLM_RE), ("E3", f"(?:{LLM_RE}|{VIT_RE})")]:
    print(s, sum(1 for n, _ in model.named_modules() if re.fullmatch(pat, n)))
proj = sum(p.numel() for n, p in model.named_parameters() if any(k in n for k in PROJ_KEYS))
print(f"projector (merger + deepstack): {proj / 1e6:.1f}M  (kỳ vọng ≈ 109M)")

In [ ]:
# (Ô dọn dẹp tạm thời đã dùng xong — đã vô hiệu hoá để "Chạy tất cả" không xoá kết quả E0. Có thể xoá ô này.)

## 6. Vòng huấn luyện

In [ ]:
# Bước 6: vòng huấn luyện (fp16 + GradScaler, grad accum, cosine warmup, checkpoint lên Drive)
import gc, math, random, yaml
from transformers import get_cosine_schedule_with_warmup

RUN_ROOT = BACKUP / "runs" if BACKUP.parent.exists() else Path("work/runs")

def free_gpu():
    for name in ("model", "pm"):
        if name in globals():
            del globals()[name]
    gc.collect(); torch.cuda.empty_cache()

def save_ckpt(pm, extra, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pm.save_pretrained(out_dir / "adapter")               # CHỈ lưu LoRA
    if extra:                                             # projector phải lưu riêng
        sd = dict(pm.named_parameters())
        torch.save({n: sd[n].detach().cpu() for n in extra}, out_dir / "projector.pt")

def train_setup(setup, train_rows=None, max_steps=None, r=None, alpha=None, prompt=None, answer_fn=None, tag=None):
    cfg = yaml.safe_load(open(f"configs/{setup.lower()}.yaml"))
    lc = cfg["lora"]
    r, alpha = r or lc["r"], alpha or lc["alpha"]
    max_steps = max_steps or cfg["max_steps"]
    bs, accum, lr = cfg["batch_size"], cfg["grad_accum"], cfg["lr"]
    tag = tag or setup
    out_dir = RUN_ROOT / tag
    random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])

    base = load_base_model()
    pm, extra, n_train = apply_setup(base, setup, r=r, alpha=alpha, dropout=lc["dropout"])
    pm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    pm.config.use_cache = False
    pm.train()

    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * max_steps), max_steps)
    scaler = torch.amp.GradScaler("cuda")

    rows = list(train_rows or splits["train"])
    order, pos = [], 0
    def next_batch():
        nonlocal order, pos
        if pos + bs > len(order):
            order, pos = random.sample(rows, len(rows)), 0
        b = order[pos:pos + bs]; pos += bs
        return collate([make_example(x, prompt, answer_fn) for x in b])

    torch.cuda.reset_peak_memory_stats()
    t0, hist = time.time(), []
    pbar = tqdm(range(1, max_steps + 1), desc=tag)
    for step in pbar:
        tot = 0.0
        for _ in range(accum):
            batch = {k: v.to("cuda") for k, v in next_batch().items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = pm(**batch).loss / accum
            if not torch.isfinite(loss):
                raise RuntimeError(f"loss không hữu hạn ở bước {step} (tràn số fp16?)")
            scaler.scale(loss).backward()
            tot += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        hist.append(tot)
        pbar.set_postfix(loss=f"{tot:.4f}", lr=f"{sched.get_last_lr()[0]:.2e}")
        if step % 100 == 0 or step == max_steps:
            save_ckpt(pm, extra, out_dir)

    info = {"name": tag, "setup": setup, "trainable_params": n_train,
            "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "train_minutes": round((time.time() - t0) / 60, 1),
            "max_steps": max_steps, "r": r, "alpha": alpha, "n_train_images": len(rows),
            "loss_first10": sum(hist[:10]) / min(10, len(hist)), "loss_last10": sum(hist[-10:]) / min(10, len(hist))}
    (out_dir / "run.json").write_text(json.dumps(info, indent=2))
    (out_dir / "loss.json").write_text(json.dumps(hist))
    print(json.dumps(info, indent=2))
    plt.figure(figsize=(8, 3)); plt.plot(hist); plt.title(f"{tag} loss"); plt.xlabel("bước"); plt.show()
    return pm, extra, info

print("checkpoint sẽ lưu ở:", RUN_ROOT)

## 7. Suy luận và chấm E1–E3

In [ ]:
# Bước 7: suy luận + chấm E1–E3 (bỏ qua setup đã có evidence, nên chạy lại cell này được nếu mất session)
import subprocess
from peft import PeftModel

def prepare_for_inference(pm, extra):
    sd = dict(pm.named_parameters())
    for n in extra:                                       # projector fp32 -> fp16 cho khớp dtype khi suy luận
        sd[n].data = sd[n].data.half()
    pm.gradient_checkpointing_disable()
    pm.config.use_cache = True
    pm.eval()
    return pm

def load_trained(tag):
    """Nạp lại từ Drive: model gốc 4-bit -> adapter LoRA -> projector (nếu có)."""
    d = RUN_ROOT / tag
    pm = PeftModel.from_pretrained(load_base_model(), d / "adapter")
    if (d / "projector.pt").exists():
        sd = torch.load(d / "projector.pt")
        own = dict(pm.named_parameters())
        for n, v in sd.items():
            own[n].data = v.to(own[n].device, torch.float16)
        print(f"đã nạp lại {len(sd)} tensor projector")
    pm.config.use_cache = True
    return pm.eval()

def score(tag, pred, info=None, split="test", coord="norm1000"):
    cmd = ["python", "scripts/score.py", "--pred", pred, "--name", tag, "--split", split, "--coord", coord,
           "--out", f"report/evidence/{tag.lower()}.json"]
    for k in ("trainable_params", "peak_vram_gb", "train_minutes"):
        if info and k in info:
            cmd += ["--meta", f"{k}={info[k]}"]
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)
    if BACKUP.parent.exists():
        (BACKUP / "evidence").mkdir(parents=True, exist_ok=True)
        shutil.copy(f"report/evidence/{tag.lower()}.json", BACKUP / "evidence")

def run_experiment(setup, tag=None, **kw):
    tag = tag or setup
    if Path(f"report/evidence/{tag.lower()}.json").exists():
        print(f"{tag}: đã có evidence, bỏ qua"); return
    free_gpu()
    run_json = RUN_ROOT / tag / "run.json"
    if run_json.exists():                                 # đã train xong trước đó -> chỉ suy luận
        info = json.loads(run_json.read_text()); pm = load_trained(tag)
    else:
        pm, extra, info = train_setup(setup, tag=tag, **kw)
        pm = prepare_for_inference(pm, extra)
    predict(pm, splits["test"], f"work/{tag.lower()}.jsonl", prompt=kw.get("prompt"))
    score(tag, f"work/{tag.lower()}.jsonl", info)
    del pm; gc.collect(); torch.cuda.empty_cache()

for s in ["E1", "E2", "E3"]:
    run_experiment(s)

In [ ]:
import os
from pathlib import Path

print("--- Các tệp tin trong thư mục work/ ---")
if Path("work").exists():
    for f in Path("work").glob("*"):
        if f.is_file():
            print(f"{f.name}: {f.stat().st_size / 1024:.2f} KB")
else:
    print("Thư mục work không tồn tại.")

print("\n--- Các tệp tin trong thư mục report/evidence/ ---")
evidence_dir = Path("report/evidence")
if evidence_dir.exists():
    for f in evidence_dir.glob("*"):
        print(f"{f.name}: {f.stat().st_size / 1024:.2f} KB")
else:
    print("Thư mục report/evidence không tồn tại.")

print("\n--- Tiến trình huấn luyện trong Drive (runs) ---")
runs_dir = Path("/content/drive/MyDrive/lab10_backup/runs")
if runs_dir.exists():
    for d in runs_dir.iterdir():
        if d.is_dir():
            run_json = d / "run.json"
            status = "Hoàn thành (có run.json)" if run_json.exists() else "Đang chạy hoặc dở dang"
            print(f"{d.name}: {status}")
else:
    print("Không tìm thấy thư mục backup trên Drive.")

In [ ]:
import os
from pathlib import Path

# Kiểm tra thư mục hiện tại
print("Thư mục làm việc hiện tại:", os.getcwd())

# Đảm bảo chúng ta đang ở trong thư mục /content/lab10
if Path("/content/lab10").exists():
    os.chdir("/content/lab10")
    print("Đã di chuyển tới: /content/lab10")
else:
    print("Thư mục /content/lab10 chưa được tạo, vui lòng chạy lại ô Clone Git.")

# Tạo lại các thư mục đích nếu chưa có để sẵn sàng chạy tiếp
Path("work").mkdir(exist_ok=True)
Path("report/evidence").mkdir(parents=True, exist_ok=True)
print("Đã kiểm tra và khởi tạo các thư mục 'work' và 'report/evidence'.")

# Kiểm tra sự tồn tại của Google Drive backup
drive_path = Path("/content/drive/MyDrive/lab10_backup")
if drive_path.exists():
    print("Đã kết nối với Google Drive và tìm thấy thư mục backup của lab10.")
else:
    print("Không tìm thấy thư mục lab10_backup trong Google Drive. Sẽ lưu tạm thời trên bộ nhớ cục bộ.")

In [ ]:
# Tải lại mã nguồn dự án nếu bị mất và di chuyển vào thư mục làm việc chính xác
import os
from pathlib import Path

if not Path("/content/lab10").exists():
    print("--- Đang tải lại mã nguồn dự án ---")
    !git clone -q https://github.com/nhat-thang/K4-Track-4-Day10-VisionLanguageModel /content/lab10

os.chdir("/content/lab10")
print("Thư mục làm việc hiện tại:", os.getcwd())

# Khởi tạo các thư mục kết quả
Path("work").mkdir(exist_ok=True)
Path("report/evidence").mkdir(parents=True, exist_ok=True)
print("Đã đảm bảo các thư mục 'work' và 'report/evidence' tồn tại.")

# Tải lại bộ dữ liệu nếu chưa có
if not Path("data").exists():
    print("--- Đang tải lại bộ dữ liệu ảnh đường phố ---")
    !python scripts/prepare_data.py --zip "https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes"

In [ ]:
# Thực hiện chạy lại từ E0 đến các thực nghiệm E1, E2, E3 để hoàn thành tiến trình
import os
import sys
import json
import time
import shutil
import re
import gc
import math
import random
import yaml
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, get_cosine_schedule_with_warmup
from scripts.labcommon import read_jsonl, CLASSES
from scripts.score import parse_output

os.chdir("/content/lab10")
sys.path.append("/content/lab10")

print("--- Đang cấu hình các tham số và hàm tiện ích ---")
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
BACKUP = Path("/content/drive/MyDrive/lab10_backup")
RUN_ROOT = BACKUP / "runs" if BACKUP.parent.exists() else Path("work/runs")
DATA = Path("data")
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}

PROMPT = (
    "Detect all road damage in this image. Classes: pothole, crack, manhole. "
    "Return ONLY a JSON list, no other text, in the form "
    '[{\"label\":\"pothole\",\"bbox_2d\":[x1,y1,x2,y2]}]. '
    "Coordinates are integers on a 0-1000 scale relative to image width and height "
    "(x1,y1 = top-left, x2,y2 = bottom-right). Return [] if there is no damage."
)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    clamp = lambda v: max(0, min(1000, int(round(v * 1000))))
    x1, y1 = clamp(xc - w / 2), clamp(yc - h / 2)
    x2, y2 = clamp(xc + w / 2), clamp(yc + h / 2)
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < 1000 else (999, 1000)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < 1000 else (999, 1000)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

def load_base_model():
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["model.visual", "lm_head"])
    m = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={"": 0})
    m.eval()
    return m

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

def build_messages(img, prompt=None):
    return [{"role": "user", "content": [{"type": "image", "image": img},
                                         {"type": "text", "text": prompt or PROMPT}]}]

def encode_example(img, prompt, answer):
    user = build_messages(img, prompt)[0]
    asst = {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    kw = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt_enc = processor.apply_chat_template([user], add_generation_prompt=True, **kw)
    full_enc = processor.apply_chat_template([user, asst], add_generation_prompt=False, **kw)
    n = prompt_enc["input_ids"].shape[1]
    ex = {k: v for k, v in full_enc.items()}
    labels = ex["input_ids"].clone()
    labels[:, :n] = -100
    ex["labels"] = labels
    return ex

NO_PAD = ("pixel_values", "image_grid_thw")
PAD_VALUE = {"input_ids": processor.tokenizer.pad_token_id, "labels": -100}

def collate(batch):
    L = max(b["input_ids"].shape[1] for b in batch)
    out = {}
    for k in batch[0]:
        if k in NO_PAD:
            out[k] = torch.cat([b[k] for b in batch], dim=0)
        else:
            out[k] = torch.cat([F.pad(b[k], (0, L - b[k].shape[1]), value=PAD_VALUE.get(k, 0)) for b in batch], dim=0)
    return out

def make_example(row, prompt=None, answer_fn=None):
    img = Image.open(DATA / row["image_path"]).convert("RGB")
    return encode_example(img, prompt or PROMPT, (answer_fn or objects_to_answer)(row["objects"]))

LLM_RE = r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
VIT_RE = r"model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.linear_fc[12])"
PROJ_KEYS = ("visual.merger", "visual.deepstack_merger_list")

def apply_setup(model, setup, r=16, alpha=32, dropout=0.05):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    pattern = LLM_RE if setup in ("E1", "E2") else f"(?:{LLM_RE}|{VIT_RE})"
    pm = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=pattern))
    extra = []
    if setup in ("E2", "E3"):
        for n, p in pm.named_parameters():
            if any(k in n for k in PROJ_KEYS):
                p.data = p.data.float()
                p.requires_grad = True
                extra.append(n)
    n_train = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    return pm, extra, n_train

def free_gpu():
    for name in ("model", "pm"):
        if name in globals():
            del globals()[name]
    gc.collect(); torch.cuda.empty_cache()

def save_ckpt(pm, extra, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pm.save_pretrained(out_dir / "adapter")
    if extra:
        sd = dict(pm.named_parameters())
        torch.save({n: sd[n].detach().cpu() for n in extra}, out_dir / "projector.pt")

def train_setup(setup, train_rows=None, max_steps=None, r=None, alpha=None, prompt=None, answer_fn=None, tag=None):
    cfg = yaml.safe_load(open(f"configs/{setup.lower()}.yaml"))
    lc = cfg["lora"]
    r, alpha = r or lc["r"], alpha or lc["alpha"]
    max_steps = max_steps or cfg["max_steps"]
    bs, accum, lr = cfg["batch_size"], cfg["grad_accum"], cfg["lr"]
    tag = tag or setup
    out_dir = RUN_ROOT / tag
    random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    base = load_base_model()
    pm, extra, n_train = apply_setup(base, setup, r=r, alpha=alpha, dropout=lc["dropout"])
    pm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    pm.config.use_cache = False
    pm.train()
    params = [p for p in pm.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * max_steps), max_steps)
    scaler = torch.amp.GradScaler("cuda")
    rows = list(train_rows or splits["train"])
    order, pos = [], 0
    def next_batch():
        nonlocal order, pos
        if pos + bs > len(order):
            order, pos = random.sample(rows, len(rows)), 0
        b = order[pos:pos + bs]; pos += bs
        return collate([make_example(x, prompt, answer_fn) for x in b])
    torch.cuda.reset_peak_memory_stats()
    t0, hist = time.time(), []
    pbar = tqdm(range(1, max_steps + 1), desc=tag)
    for step in pbar:
        tot = 0.0
        for _ in range(accum):
            batch = {k: v.to("cuda") for k, v in next_batch().items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = pm(**batch).loss / accum
            scaler.scale(loss).backward()
            tot += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        hist.append(tot)
        pbar.set_postfix(loss=f"{tot:.4f}")
        if step % 100 == 0 or step == max_steps:
            save_ckpt(pm, extra, out_dir)
    info = {"name": tag, "setup": setup, "trainable_params": n_train,
            "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "train_minutes": round((time.time() - t0) / 60, 1),
            "max_steps": max_steps, "r": r, "alpha": alpha, "n_train_images": len(rows),
            "loss_first10": sum(hist[:10]) / min(10, len(hist)), "loss_last10": sum(hist[-10:]) / min(10, len(hist))}
    (out_dir / "run.json").write_text(json.dumps(info, indent=2))
    return pm, extra, info

@torch.inference_mode()
def predict(model, rows, out_path, bs=4, max_new_tokens=256, prompt=None):
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    done = {r["image_id"] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r["image_id"] not in done]
    with open(out_path, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), bs), desc="Predicting"):
            batch = todo[i:i + bs]
            msgs = [build_messages(Image.open(DATA / r["image_path"]).convert("RGB"), prompt) for r in batch]
            inputs = processor.apply_chat_template(
                msgs, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            texts = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for r, t in zip(batch, texts):
                f.write(json.dumps({"image_id": r["image_id"], "raw": t}, ensure_ascii=False) + "\n")
            f.flush()

def prepare_for_inference(pm, extra):
    sd = dict(pm.named_parameters())
    for n in extra:
        sd[n].data = sd[n].data.half()
    pm.gradient_checkpointing_disable()
    pm.config.use_cache = True
    pm.eval()
    return pm

def load_trained(tag):
    d = RUN_ROOT / tag
    pm = PeftModel.from_pretrained(load_base_model(), d / "adapter")
    if (d / "projector.pt").exists():
        sd = torch.load(d / "projector.pt")
        own = dict(pm.named_parameters())
        for n, v in sd.items():
            own[n].data = v.to(own[n].device, torch.float16)
    pm.config.use_cache = True
    return pm.eval()

def score(tag, pred, info=None, split="test", coord="norm1000"):
    import subprocess
    cmd = ["python", "scripts/score.py", "--pred", pred, "--name", tag, "--split", split, "--coord", coord,
           "--out", f"report/evidence/{tag.lower()}.json"]
    for k in ("trainable_params", "peak_vram_gb", "train_minutes"):
        if info and k in info:
            cmd += ["--meta", f"{k}={info[k]}"]
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

def run_experiment(setup, tag=None, **kw):
    tag = tag or setup
    if Path(f"report/evidence/{tag.lower()}.json").exists():
        print(f"{tag}: đã có evidence, bỏ qua"); return
    free_gpu()
    run_json = RUN_ROOT / tag / "run.json"
    if run_json.exists():
        info = json.loads(run_json.read_text()); pm = load_trained(tag)
    else:
        pm, extra, info = train_setup(setup, tag=tag, **kw)
        pm = prepare_for_inference(pm, extra)
    predict(pm, splits["test"], f"work/{tag.lower()}.jsonl", prompt=kw.get("prompt"))
    score(tag, f"work/{tag.lower()}.jsonl", info)
    del pm; gc.collect(); torch.cuda.empty_cache()

# --- CHẠY THỬ NGHIỆM ---
# 1. Chạy Zero-Shot E0
if not Path("report/evidence/e0.json").exists():
    print("--- Đang chạy Zero-Shot E0 ---")
    free_gpu()
    base_model = load_base_model()
    predict(base_model, splits["test"], "work/e0.jsonl")
    score("E0", "work/e0.jsonl")
    del base_model; gc.collect(); torch.cuda.empty_cache()

# 2. Chạy E1, E2, E3
print("--- Đang chạy các thực nghiệm E1, E2, E3 ---")
for s in ["E1", "E2", "E3"]:
    run_experiment(s)


In [ ]:
import os
from pathlib import Path
import json

evidence_dir = Path("report/evidence")
print("=== BẢNG ĐIỂM F1 THỰC NGHIỆM CHI TIẾT ===")
for exp in ["e0", "e1", "e2", "e3", "e4"]:
    f_path = evidence_dir / f"{exp}.json"
    if f_path.exists():
        try:
            data = json.loads(f_path.read_text())
            # Lấy F1 từ data['micro']['f1'] hoặc data['metrics']['f1']
            f1_val = None
            if "micro" in data and "f1" in data["micro"]:
                f1_val = data["micro"]["f1"]
            elif "metrics" in data and "f1" in data["metrics"]:
                f1_val = data["metrics"]["f1"]

            if f1_val is not None:
                print(f"[✓] {exp.upper()}: Hoàn thành | F1-Score: {f1_val * 100:.2f}%")
            else:
                print(f"[✓] {exp.upper()}: Hoàn thành | F1-Score: Không tìm thấy trường điểm")
        except Exception as e:
            print(f"[!] {exp.upper()}: Lỗi khi đọc file ({e})")
    else:
        print(f"[ ] {exp.upper()}: Chưa hoàn thành (Đang chờ xử lý...)")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# Tiến hành thực nghiệm E4 (Ablation Study) bằng cách giảm LoRA Rank xuống r=8, alpha=16 dựa trên cấu hình E1
import os
from pathlib import Path

os.chdir("/content/lab10")

tag_e4 = "E4"
if Path(f"report/evidence/{tag_e4.lower()}.json").exists():
    print(f"{tag_e4}: Đã có file kết quả, bỏ qua huấn luyện lại.")
else:
    print(f"--- Đang bắt đầu huấn luyện thực nghiệm Ablation {tag_e4} (r=8, alpha=16) ---")
    # Chạy huấn luyện và đánh giá E4
    run_experiment(setup="E1", tag=tag_e4, r=8, alpha=16)


In [ ]:
# Nâng cấp torchao để tương thích với PEFT khi cấu hình LoRA trên Vision modules (E3)
!pip install -q "torchao>=0.16.0"

import os
from pathlib import Path

os.chdir("/content/lab10")

tag_e3 = "E3"
if Path(f"report/evidence/{tag_e3.lower()}.json").exists():
    print(f"{tag_e3}: Đã có file kết quả, bỏ qua huấn luyện lại.")
else:
    print(f"--- Đang bắt đầu huấn luyện thực nghiệm {tag_e3} (LLM + Vision + Projector) ---")
    # Gọi hàm để chạy huấn luyện và đánh giá E3
    run_experiment(setup="E3", tag=tag_e3)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9

In [ ]:
# ===== BƯỚC 1: KIỂM TRA TRẠNG THÁI + CỨU BẰNG CHỨNG (chạy lại an toàn, không ghi đè) =====
import os, json, glob, hashlib, shutil, subprocess, time, importlib
REPO="/content/lab10"; BACKUP="/content/drive/MyDrive/lab10_backup"
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
def hsize(p):
    n=os.path.getsize(p)
    for u in ["B","KB","MB","GB"]:
        if n<1024: return f"{n:.0f}{u}"
        n/=1024
    return f"{n:.1f}TB"
def md5(p):
    h=hashlib.md5()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()
def sh(c):
    try: return subprocess.run(c,shell=True,capture_output=True,text=True,timeout=60).stdout.strip()
    except Exception as e: return f"ERR {e}"
print("="*60,"\n[0] Drive/repo")
print("Drive mounted:",os.path.ismount("/content/drive"),"| backup:",os.path.isdir(BACKUP),"| repo:",os.path.isdir(REPO))
if os.path.isdir(REPO):
    print("remote:",sh(f"cd {REPO} && git remote -v | head -1")); print("HEAD:",sh(f"cd {REPO} && git log --oneline -1"))
    print("status:\n"+(sh(f"cd {REPO} && git status --short | head -30") or "  (sạch)"))
    print("data/:",sorted(os.listdir(f"{REPO}/data"))[:20] if os.path.isdir(f"{REPO}/data") else "KHÔNG CÓ")
print("="*60,"\n[1] evidence")
def show_ev(p):
    try: d=json.load(open(p))
    except Exception as e: print(" ",p,"LỖI",e); return
    m=d.get("meta",{}) or {}
    print(f"\n ▶ {p} ({hsize(p)})")
    print("   micro:",d.get("micro")); print("   ci95:",d.get("f1_ci95"),"| mIoU:",d.get("mean_iou"),"| valid:",d.get("parse_valid_rate"),"| missing:",d.get("missing_predictions"))
    print("   per_class f1:",{k:(v.get("f1") if isinstance(v,dict) else v) for k,v in (d.get("per_class") or {}).items()})
    print("   meta:",json.dumps(m,ensure_ascii=False))
    iss=[]
    for k in ["trainable_params","peak_vram_gb","train_minutes"]:
        try:
            if not float(m.get(k,0))>0: iss.append(k+"<=0/thiếu")
        except Exception: iss.append(k+" không phải số")
    if m.get("split",d.get("split"))!="test": iss.append(f"split={m.get('split',d.get('split'))}")
    if d.get("missing_predictions") not in (0,None): iss.append(f"missing={d.get('missing_predictions')}")
    print("   CHECK:","OK" if not iss else "⚠ "+"; ".join(iss))
evl=sorted(glob.glob(f"{REPO}/report/evidence/*.json")); evd=sorted(glob.glob(f"{BACKUP}/**/*.json",recursive=True))
print("Local:",[os.path.basename(p) for p in evl] or "KHÔNG CÒN")
for p in evl: show_ev(p)
print("Drive json:",[os.path.relpath(p,BACKUP) for p in evd] or "KHÔNG CÓ")
if not evl:
    for p in evd:
        if "evidence" in p: show_ev(p)
print("="*60,"\n[2] jsonl")
def show_jl(p):
    n=sum(1 for l in open(p) if l.strip())
    print(f"  {p} {hsize(p)} dòng={n}\n    dòng1: {open(p).readline().strip()[:200]}")
wl=sorted(glob.glob(f"{REPO}/work/*.jsonl")); wd=sorted(glob.glob(f"{BACKUP}/**/*.jsonl",recursive=True))
print("Local:"); [show_jl(p) for p in wl] or print("  KHÔNG CÒN")
print("Drive:"); [show_jl(p) for p in wd] or print("  KHÔNG CÓ")
print("="*60,"\n[3] Drive tree")
if os.path.isdir(BACKUP):
    for root,dirs,files in os.walk(BACKUP):
        d=root[len(BACKUP):].count(os.sep)
        if d>3: continue
        print("  "*d+(os.path.basename(root) or BACKUP)+"/")
        for fn in sorted(files)[:12]: print("  "*(d+1)+f"{fn} ({hsize(os.path.join(root,fn))})")
        if len(files)>12: print("  "*(d+1)+f"... +{len(files)-12}")
    for rj in sorted(glob.glob(f"{BACKUP}/runs/*/run.json")): print(f"\n ▶ {os.path.relpath(rj,BACKUP)}:",open(rj).read()[:700])
print("="*60,"\n[4] GPU/libs")
print(sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader") or "Không có GPU")
for lib in ["torch","transformers","peft","bitsandbytes","accelerate","torchao","trl"]:
    try: print(f"  {lib:13s}",importlib.import_module(lib).__version__)
    except Exception as e: print(f"  {lib:13s} KHÔNG CÓ")
print("="*60,"\n[5] Session:",{n:(n in globals()) for n in ["run_experiment","load_trained","prepare_for_inference","score","build_messages","parse_output","model"]})
print("="*60,"\n[6] Backup (không ghi đè)")
def safe_copy(s,dd):
    os.makedirs(dd,exist_ok=True); t=os.path.join(dd,os.path.basename(s))
    if os.path.exists(t):
        if md5(t)==md5(s): return "giống, bỏ qua"
        b,e=os.path.splitext(t); t=f"{b}.{time.strftime('%Y%m%d-%H%M%S')}{e}"
    shutil.copy2(s,t); return "-> "+os.path.relpath(t,BACKUP)
for p in evl: print(" ",os.path.basename(p),safe_copy(p,f"{BACKUP}/report/evidence"))
for p in wl: print(" ",os.path.basename(p),safe_copy(p,f"{BACKUP}/work"))
if not (evl or wl): print("  Không có gì local để chép.")


In [ ]:
# Bước 1b: quét toàn Drive tìm evidence/jsonl/adapter/projector của E2–E4 (chỉ đọc)
import subprocess
print(subprocess.run(r'''find /content/drive/MyDrive -maxdepth 7 \( -iname "e[0-9]*.json*" -o -iname "*projector*" -o -iname "*merger*" -o -iname "*deepstack*" -o -iname "adapter_config.json" -o -iname "run.json" -o -iname "lab_day10*.ipynb" -o -ipath "*runs/E*" -prune \) -not -path "*/.Trash*" -printf "%TY-%Tm-%Td %TH:%TM  %10s  %p\n" 2>/dev/null | sort''', shell=True, capture_output=True, text=True).stdout or 'Không tìm thấy gì')


In [ ]:
# Bước 1c: clone lại repo nếu mất (bỏ qua nếu đã có) + xem score.py cần gì
import os, subprocess
REPO = "/content/lab10"
def sh(c): r = subprocess.run(c, shell=True, capture_output=True, text=True); return (r.stdout + r.stderr).strip()
if not os.path.isdir(REPO):
    print(sh(f"git clone -q https://github.com/nhat-thang/K4-Track-4-Day10-VisionLanguageModel {REPO}"))
os.chdir(REPO)
print("HEAD:", sh("git log --oneline -3"))
print(sh("find . -path ./.git -prune -o -type f -print | grep -v -E '^./(data|work)/' | sort | head -80"))
print("\n--- score.py --help ---\n" + sh("python scripts/score.py --help"))
print("\n--- score.py: đường dẫn dữ liệu/nhãn ---\n" + sh("grep -n -E 'data/|DATA|annot|label|json|csv|path|Path\\(' scripts/score.py | head -40"))
print("\n--- GUIDE: tải dữ liệu ---\n" + sh("grep -n -i -E 'kaggle|download|tải|data/|prepare|dataset_version' GUIDE.md | head -40"))
print("\n--- check_submission.py: các mục kiểm tra ---\n" + sh("grep -n -E 'def |evidence|figure|REPORT|meta|<điền' scripts/check_submission.py | head -60"))


In [9]:
# Bước 1d: chuẩn bị dữ liệu (bỏ qua nếu đã có) — khoá Kaggle lấy từ Colab Secrets, không ghi vào code
import os, json, subprocess, shutil
REPO = "/content/lab10"; BACKUP = "/content/drive/MyDrive/lab10_backup"
os.chdir(REPO)
SPL = f"{REPO}/data/splits"
if not os.path.exists(f"{SPL}/meta.json"):
    from google.colab import userdata
    try:    # token kiểu mới (KGAT_...)
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:   # hoặc cặp username/key kiểu cũ
        for k in ["KAGGLE_USERNAME", "KAGGLE_KEY"]: os.environ[k] = userdata.get(k)
    r = subprocess.run("pip -q install -U kaggle && python scripts/prepare_data.py --kaggle", shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-2000:])
meta = json.load(open(f"{SPL}/meta.json"))
print("meta.json:", {k: v for k, v in meta.items() if not isinstance(v, (list, dict))})
print("dataset_version khớp d10-21e4070301:", meta.get("dataset_version") == "d10-21e4070301")
for s in ["train", "val", "test"]:
    print(s, sum(1 for l in open(f"{SPL}/{s}.jsonl") if l.strip()), "ảnh")
print("số file ảnh:", len(os.listdir(f"{REPO}/data/images")) if os.path.isdir(f"{REPO}/data/images") else 0)
# backup splits (nhỏ) sang Drive để lần sau không cần tải lại để chấm điểm
dst = f"{BACKUP}/data_splits"; os.makedirs(dst, exist_ok=True)
for fn in os.listdir(SPL):
    if not os.path.exists(f"{dst}/{fn}"): shutil.copy2(f"{SPL}/{fn}", dst)
print("Đã backup splits:", sorted(os.listdir(dst)))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.1 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
License(s): MIT

dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}


  0%|          | 0.00/185M [00:00<?, ?B/s]
  1%|          | 2.00M/185M [00:00<00:11, 16.1MB/s]
  3%|▎         | 5.00M/185M [00:00<00:08, 21.9MB/s]
  4%|▍         | 8.00M/185M [00:00<00:07, 23.9MB/s]
  6%|▋         | 12.0M/185M [00:00<00:06, 29.3MB/s]
 10%|▉         | 18.0M/185M [00:00<00:04, 38.7MB/s]
 13%|█▎        | 24.0M/185M [00:00<00:03, 45.0MB/s]
 16%|█▌        | 29.0M/185M [00:00<00:03, 44.1MB/s]
 19%|█▉        | 35.0M/185M [00:00<00:03, 45.6MB/s]
 23%|██▎       | 42.0M/185M [00:01<00:02, 51.7MB/s]
 27%|██▋       | 50.0M/185M [00

In [10]:
# Bước 2a: chấm lại E0/E1 từ jsonl trên Drive + Câu 3 (coord pixel) — bỏ qua nếu đã có, chép sang Drive
import os, json, shutil, subprocess
REPO = "/content/lab10"; BACKUP = "/content/drive/MyDrive/lab10_backup"
os.chdir(REPO); os.makedirs("work", exist_ok=True)
for t in ["e0", "e1"]:                       # lấy predictions từ Drive về work/
    if not os.path.exists(f"work/{t}.jsonl"): shutil.copy2(f"{BACKUP}/{t}.jsonl", f"work/{t}.jsonl")
run1 = json.load(open(f"{BACKUP}/runs/E1/run.json"))
jobs = [  # (pred, name, out, coord, meta)
    ("work/e0.jsonl", "E0", "report/evidence/e0.json", "norm1000", {"trainable_params": 0, "train_minutes": 0}),
    ("work/e1.jsonl", "E1", "report/evidence/e1.json", "norm1000",
        {k: run1[k] for k in ["trainable_params", "peak_vram_gb", "train_minutes", "r", "alpha", "max_steps"]}),
    ("work/e1.jsonl", "E1_coord_pixel", "report/extra/coord_pixel_check.json", "pixel", {"note": "chấm lại e1.jsonl với --coord pixel"}),
]
for pred, name, out, coord, meta in jobs:
    if os.path.exists(out):
        print(f"{out} đã có — bỏ qua"); continue
    cmd = ["python", "scripts/score.py", "--pred", pred, "--name", name, "--split", "test",
           "--coord", coord, "--out", out] + sum([["--meta", f"{k}={v}"] for k, v in meta.items()], [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout, r.stderr[-1500:])
for out in [j[2] for j in jobs]:
    d = json.load(open(out)); m = d["micro"]
    print(f"{os.path.basename(out):24s} F1={m['f1']:.3f} CI={d['f1_ci95'][0]:.3f}-{d['f1_ci95'][1]:.3f} "
          f"valid={d['parse_valid_rate']:.3f} FP/img={d.get('fp_per_image', float('nan')):.2f} mIoU={d['mean_iou']:.3f} "
          f"missing={d['missing_predictions']} split={d['split']} coord={d['coord']} ver={d['dataset_version']} meta={d['meta']}")
# so với bản e1.json cũ trên Drive
old = json.load(open(f"{BACKUP}/evidence/e1.json")); new = json.load(open("report/evidence/e1.json"))
print("e1 mới vs e1 cũ trên Drive: F1", round(new["micro"]["f1"], 4), "vs", round(old["micro"]["f1"], 4))
# backup
for sub in ["report/evidence", "report/extra"]:
    os.makedirs(f"{BACKUP}/{sub}", exist_ok=True)
    for fn in os.listdir(sub):
        if fn.endswith(".json"): shutil.copy2(f"{sub}/{fn}", f"{BACKUP}/{sub}/{fn}")
print("Drive report/:", {s: sorted(os.listdir(f"{BACKUP}/{s}")) for s in ["report/evidence", "report/extra"]})


E0 [test] n=311 valid=0.952 P=0.144 R=0.077 F1=0.100 CI95=0.074-0.131 meanIoU=0.600 FP/img=0.86
   pothole  P=0.264 R=0.116 F1=0.161
   crack    P=0.032 R=0.020 F1=0.025
   manhole  P=0.352 R=0.153 F1=0.214
 
E1 [test] n=311 valid=1.000 P=0.276 R=0.244 F1=0.259 CI95=0.225-0.295 meanIoU=0.679 FP/img=1.20
   pothole  P=0.155 R=0.074 F1=0.101
   crack    P=0.168 R=0.197 F1=0.182
   manhole  P=0.698 R=0.454 F1=0.550
 
E1_coord_pixel [test] n=311 valid=1.000 P=0.000 R=0.000 F1=0.000 CI95=0.000-0.000 meanIoU=0.000 FP/img=1.66
   pothole  P=0.000 R=0.000 F1=0.000
   crack    P=0.000 R=0.000 F1=0.000
   manhole  P=0.000 R=0.000 F1=0.000
 
e0.json                  F1=0.100 CI=0.074-0.131 valid=0.952 FP/img=0.86 mIoU=0.600 missing=0 split=test coord=norm1000 ver=d10-21e4070301 meta={'trainable_params': 0, 'train_minutes': 0}
e1.json                  F1=0.259 CI=0.225-0.295 valid=1.000 FP/img=1.20 mIoU=0.679 missing=0 split=test coord=norm1000 ver=d10-21e4070301 meta={'trainable_params': 33030144

In [11]:
# Bước 5: đếm tham số ViT / projector / LLM (meta device, không tải trọng số) + số tham số LoRA theo shape
import os, re, json, torch
from transformers import AutoConfig, AutoModelForImageTextToText
from accelerate import init_empty_weights
OUT = "/content/lab10/report/extra/param_counts.json"; BACKUP = "/content/drive/MyDrive/lab10_backup"
cfg = AutoConfig.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
with init_empty_weights():
    m = AutoModelForImageTextToText.from_config(cfg)
groups = {"vit_backbone": 0, "projector_merger": 0, "projector_deepstack": 0, "llm": 0}
for n, p in m.named_parameters():
    if "visual.merger" in n: groups["projector_merger"] += p.numel()
    elif "visual.deepstack_merger_list" in n: groups["projector_deepstack"] += p.numel()
    elif "visual" in n: groups["vit_backbone"] += p.numel()
    else: groups["llm"] += p.numel()
groups["projector_total"] = groups["projector_merger"] + groups["projector_deepstack"]
groups["total"] = sum(p.numel() for p in m.parameters())
groups["tied_word_embeddings"] = bool(getattr(cfg, "tie_word_embeddings", getattr(cfg.text_config, "tie_word_embeddings", None)))
# LoRA: mỗi Linear (in,out) thêm r*(in+out)
LLM_T = re.compile(r"language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$")
VIT_T = re.compile(r"visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))$")
def lora(pat, r):
    mods = [(n, mod) for n, mod in m.named_modules() if isinstance(mod, torch.nn.Linear) and pat.search(n)]
    return len(mods), sum(r * (mod.in_features + mod.out_features) for _, mod in mods)
for r in [8, 16]:
    nl, pl = lora(LLM_T, r); nv, pv = lora(VIT_T, r)
    groups[f"lora_llm_r{r}"] = {"modules": nl, "params": pl}
    groups[f"lora_vit_r{r}"] = {"modules": nv, "params": pv}
groups["calc_E1_r16"] = groups["lora_llm_r16"]["params"]
groups["calc_E2"] = groups["calc_E1_r16"] + groups["projector_total"]
groups["calc_E3"] = groups["calc_E2"] + groups["lora_vit_r16"]["params"]
groups["calc_E4_r8"] = groups["lora_llm_r8"]["params"]
os.makedirs(os.path.dirname(OUT), exist_ok=True); json.dump(groups, open(OUT, "w"), indent=2)
os.makedirs(f"{BACKUP}/report/extra", exist_ok=True); import shutil; shutil.copy2(OUT, f"{BACKUP}/report/extra/")
for k, v in groups.items(): print(f"{k:22s}", f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else v)


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

vit_backbone           306,242,560
projector_merger       27,271,680
projector_deepstack    81,833,472
llm                    4,411,424,256
projector_total        109,105,152
total                  4,826,771,968
tied_word_embeddings   True
lora_llm_r8            {'modules': 252, 'params': 16515072}
lora_vit_r8            {'modules': 96, 'params': 3145728}
lora_llm_r16           {'modules': 252, 'params': 33030144}
lora_vit_r16           {'modules': 96, 'params': 6291456}
calc_E1_r16            33,030,144
calc_E2                142,135,296
calc_E3                148,426,752
calc_E4_r8             16,515,072


In [12]:
# Bước 1e: quét rộng toàn Drive (kể cả Thùng rác, Shared drives) tìm kết quả E2–E4 (chỉ đọc)
import subprocess
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print("Thư mục gốc /content/drive:", sh("ls -la /content/drive"))
print("\n--- file sửa từ 12/10, bỏ qua dữ liệu ảnh ---")
print(sh("find /content/drive -xdev -type f -newermt 2026-10-12 -not -iname '*.jpg' -not -iname '*.png' -not -path '*/Colab Notebooks/*' -printf '%TY-%Tm-%Td %TH:%TM %12s  %p\\n' 2>/dev/null | sort | head -100") or "Không có")
print("\n--- mọi .pt / .safetensors / .jsonl trên Drive ---")
print(sh("find /content/drive -xdev -type f \\( -name '*.pt' -o -name '*.safetensors' -o -name '*.jsonl' \\) -printf '%TY-%Tm-%Td %TH:%TM %12s  %p\\n' 2>/dev/null | sort") or "Không có")
print("\n--- runtime hiện tại có còn work/runs không ---")
print(sh("ls -la /content/lab10/work/runs 2>&1 | head"))


Thư mục gốc /content/drive: total 24
drwx------  5 root root 4096 Oct 14 05:24 .
drwxr-xr-x  1 root root 4096 Oct 14 05:27 ..
dr-x------  4 root root 4096 Oct 14 05:24 .Encrypted
drwx------ 16 root root 4096 Oct 14 05:24 MyDrive
dr-x------  2 root root 4096 Oct 14 05:24 .shortcut-targets-by-id
drwx------  5 root root 4096 Oct 14 05:24 .Trash-0

--- file sửa từ 12/10, bỏ qua dữ liệu ảnh ---
2026-10-13 09:37          100  /content/drive/MyDrive/lab10_backup/_smoke.jsonl
2026-10-13 09:49        43752  /content/drive/MyDrive/lab10_backup/e0.jsonl
2026-10-13 10:51         1293  /content/drive/MyDrive/lab10_backup/runs/E1/adapter/adapter_config.json
2026-10-13 10:51    132195448  /content/drive/MyDrive/lab10_backup/runs/E1/adapter/adapter_model.safetensors
2026-10-13 10:51          263  /content/drive/MyDrive/lab10_backup/runs/E1/run.json
2026-10-13 10:51         5174  /content/drive/MyDrive/lab10_backup/runs/E1/adapter/README.md
2026-10-13 10:51         5974  /content/drive/MyDrive/lab10_ba